#STURM-Flood

In [1]:
#@title Imports

import random
import os
import sys
import zipfile
import ee
import time

from dataclasses import dataclass
from pathlib import Path
from google.colab import auth, drive

In [2]:
#@title Setup

root_path = "/content/drive/MyDrive/MSc/STURM-fusion"  #@param {type:"string", multiline:true}
mount_point = "/content/drive"  #@param {type:"string"}
mount_drive = True  #@param {type:"boolean"}
clone_repo = False  #@param {type:"boolean"}
reset_export = False  #@param {type:"boolean"}
gee_export = False  #@param {type:"boolean"}
push_to_HF = False  #@param {type:"boolean"}
cancle_gee_tasks = False  #@param {type:"boolean"}
gee_project = "243624085884"  #@param {type:"string"}

if not mount_drive and not clone_repo:
    raise ValueError("Either mount_drive or clone_repo must be True.")

repo_url = "https://github.com/TAX2310/STURM-fusion.git"

if mount_drive:
    drive.mount(mount_point)
    project_root = os.path.join(root_path, "STURM-fusion") if clone_repo else root_path
else:
    project_root = "STURM-fusion"

if clone_repo and not os.path.exists(project_root):
    !git clone {repo_url} "{project_root}"

project_root = Path(project_root)
assert project_root.exists(), f"Repo not found at {project_root}. Enable clone_repo, mount_drive, or fix root_path."

if gee_export:
    ee.Authenticate(force=True, auth_mode="notebook")
    ee.Initialize(project=gee_project)

sys.path.append(str(project_root))

from src.config import CFG
cfg = CFG()
cfg.ROOT = project_root
cfg.DRIVE_ROOT = Path(mount_point) / "MyDrive" if mount_drive else project_root
cfg.GEE_PROJECT = gee_project

print("ROOT:", cfg.ROOT)
print("DRIVE_ROOT:", cfg.DRIVE_ROOT)
print("GEE project:", cfg.GEE_PROJECT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/MSc/STURM-fusion
DRIVE_ROOT: /content/drive/MyDrive
GEE project: 243624085884


In [3]:
from src.gee.tasks import cancel_all_tasks
from src.util.io import clear_export_folder, create_dataset_structure, save_dataframe_to_csv, zip_dataset
from src.data.sturm_flood import download_and_extract
from src.gee.tasks import has_active_tasks
from src.pipeline.matching import process_csv
from src.pipeline.export import export_all_s1_images
from src.pipeline.assemble import assemble_dataset
from src.pipeline.preprocessing import preprocessing_s1_pipeline, preprocessing_s2_pipeline
from src.pipeline.validation import validate_files, validate_nan_files, remove_bad_nan_files, validate_dataset
from src.util.metrics import check_image_shapes, get_band_min_max, get_band_percentiles, get_max_time_difference_with_row
from src.hugging_face.push_dataset import push_zip_to_hf

In [4]:
if cancle_gee_tasks:
  cancel_all_tasks()

In [5]:
if reset_export:
    clear_export_folder(cfg)

create_dataset_structure(cfg)

Dataset structure created:
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-flood
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset/S1
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset/S2
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset/floodmaps
 - /content/drive/MyDrive/MSc/STURM-fusion/STURM-fusion-24h/Dataset/metadata


In [6]:
#@title Download STURM-Flood

download_and_extract(cfg)

Zip already exists or dataset present, skipping download.
Dataset already extracted, skipping unzip.


In [7]:
if gee_export and not has_active_tasks():
    print("No active GEE tasks, starting")
    images, df_fusion = process_csv(cfg.OLD_S2_METADATA_CSV, cfg, verbose=True)
    print('')
    print(len(images), "images processed.")

    save_dataframe_to_csv(df_fusion, cfg.NEW_METADATA_CSV)
    print('New Metadata saved')

    if len(images) == 1970:
        export_all_s1_images(images, cfg)
        print('S1 images exported')
else:
    print("Active GEE tasks detected. Please wait for them to finish before startingt. ⏳")

Active GEE tasks detected. Please wait for them to finish before startingt. ⏳


In [8]:
while not validate_files(cfg):
    assemble_dataset(cfg)
    print('Dataset assembled')

    preprocessing_s1_pipeline(cfg)
    print('S1 preprocessing done')

    preprocessing_s2_pipeline(cfg)
    print('S2 preprocessing done')

    time.sleep(120)

preprocessing_s1_pipeline(cfg)
print('S1 preprocessing done')

preprocessing_s2_pipeline(cfg)
print('S2 preprocessing done')


Validation results:
Total rows: 1970
Missing S2: 0
Missing S1: 0
Dataset complete
Skipping (already processed): EMSR470_AOI01_38_14_2_1.tif
Skipping (already processed): EMSR470_AOI01_38_15_1_2.tif
Skipping (already processed): EMSR470_AOI01_38_18_1_2.tif
Skipping (already processed): EMSR470_AOI01_38_16_1_1.tif
Skipping (already processed): EMSR470_AOI01_38_16_1_2.tif
Skipping (already processed): EMSR470_AOI01_38_17_1_2.tif
Skipping (already processed): EMSR470_AOI01_38_17_1_1.tif
Skipping (already processed): EMSR470_AOI01_38_17_2_2.tif
Skipping (already processed): EMSR470_AOI01_38_18_1_1.tif
Skipping (already processed): EMSR470_AOI01_39_07_1_1.tif
Skipping (already processed): EMSR470_AOI01_39_06_2_2.tif
Skipping (already processed): EMSR470_AOI01_38_18_2_1.tif
Skipping (already processed): EMSR470_AOI01_39_09_1_2.tif
Skipping (already processed): EMSR470_AOI01_39_09_1_1.tif
Skipping (already processed): EMSR470_AOI01_39_07_1_2.tif
Skipping (already processed): EMSR470_AOI01_39_

In [9]:
if not validate_nan_files(cfg):
    df = remove_bad_nan_files(cfg)

Number of bad files: 1
Some files have high NaN ratio:
 - EMSR501_AOI01_08_01_1_2.tif
Number of bad files: 1
Removed: EMSR501_AOI01_08_01_1_2.tif
Removed 1 rows from metadata CSV


In [10]:
if validate_dataset(cfg):

    print('Starting inspection Pipeline')

    result = check_image_shapes(cfg.NEW_S1_PATH, cfg.NEW_S2_PATH)
    print(result)

    get_band_min_max(cfg.NEW_S1_PATH)
    get_band_percentiles(cfg.NEW_S1_PATH)

    max_s2_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel2_timestamp")

    print(f"Max S2 time difference: {max_s2_diff['time_diff_hours']:.2f} hours")

    max_s1_diff = get_max_time_difference_with_row(cfg.NEW_METADATA_CSV, sentinel_timestamp="sentinel1_timestamp")

    print(f"Max S1 time difference: {max_s1_diff['time_diff_hours']:.2f} hours")


Validation results:
Total rows: 1969
Missing S2: 0
Missing S1: 0
Dataset complete
All files marked as preprocessed
Number of bad files: 0
All files have acceptable NaN ratio
Starting inspection Pipeline

Shape check results:
Checked: 1969 files
Missing in dir2: 0
Mismatched shapes: 0
{'missing': [], 'mismatches': []}

Per-band stats:
Band 1: min=-9.553, max=8.221
Band 2: min=-10.599, max=11.245

Per-band percentiles:

Band 1:
  P1: -2.695
  P5: -1.710
  P50: 0.093
  P95: 1.445
  P99: 2.102

Band 2:
  P1: -2.673
  P5: -1.685
  P50: 0.095
  P95: 1.431
  P99: 2.050
Max time difference: 33.86 hours
Unnamed: 0                                    1138
ems_code                                   EMSR470
aoi_code                                     AOI01
floodmap_id              EMSR470_AOI01_DEL_PRODUCT
event_type                             Flash flood
country                                       Togo
tile_id                EMSR470_AOI01_04_27_1_2.tif
floodmap_date                  2020-10-1

/content/drive/MyDrive/MSc/STURM-fusion/src/util/metrics.py:110: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")
/content/drive/MyDrive/MSc/STURM-fusion/src/util/metrics.py:111: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df[sentinel_timestamp] = pd.to_datetime(df[sentinel_timestamp], dayfirst=True, errors="coerce")
/content/drive/MyDrive/MSc/STURM-fusion/src/util/metrics.py:110: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["floodmap_date"] = pd.to_datetime(df["floodmap_date"], dayfirst=True, errors="coerce")


In [11]:
if push_to_HF and validate_dataset(cfg):
    zip_dataset(cfg)
    print('Dataset zipped')
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")

    from huggingface_hub import login

    login(token=HF_TOKEN)

    url = push_zip_to_hf(zip_path=cfg.NEW_ZIP_PATH,repo_id=cfg.HF_REPO_ID,path_in_repo="Dataset.zip",private=False,)
    print(url)